# Global Earthquake Analytics & Seismic Pattern Detection
## Visual Exploration & Spatial Clustering Analysis

This notebook provides reproducible visual analytics for global earthquake patterns using aggregate outputs produced by the Apache Spark processing pipeline and MLlib spherical K-Means clustering (k=8).

**Data Source:** USGS Global Earthquake Catalog (2016–2026, 289,695 processed events).


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

# Use clean styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
os.makedirs('../output/figures', exist_ok=True)
print('Libraries loaded successfully.')


### 1. Optimal Cluster Evaluation (k-Selection: Elbow & Silhouette)
We evaluate candidate k values between 4 and 12 using spherical coordinates (x, y, z) on unit sphere, standardized depth, and magnitude.


In [ ]:
k_df = pd.read_csv('../output/k_selection.csv')
print(k_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
ax1.plot(k_df['k'], k_df['wssse'], marker='o', color='#8e44ad', linewidth=2, markersize=6)
ax1.set_title('Elbow Method (WSSSE vs. k)', fontsize=12, fontweight='bold', pad=10)
ax1.set_xlabel('Number of Clusters (k)', fontsize=11)
ax1.set_ylabel('Within-Set Sum of Squared Errors', fontsize=11)
ax1.set_xticks(k_df['k'])
ax1.grid(True, linestyle='--', alpha=0.7)

ax2.plot(k_df['k'], k_df['silhouette'], marker='s', color='#16a085', linewidth=2, markersize=6)
ax2.axvline(x=8, color='#e74c3c', linestyle=':', label='Chosen k = 8')
ax2.set_title('Silhouette Score vs. k', fontsize=12, fontweight='bold', pad=10)
ax2.set_xlabel('Number of Clusters (k)', fontsize=11)
ax2.set_ylabel('Silhouette Coefficient', fontsize=11)
ax2.set_xticks(k_df['k'])
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


### 2. Global Spatial Scatter Map (Color-Coded by Assigned Cluster)
Visualizes 20,000 sampled earthquakes classified into 8 MLlib clusters.


In [ ]:
pts = pd.read_csv('../output/points_sample.csv')
fig, ax = plt.subplots(figsize=(14, 7), dpi=150)
scatter = ax.scatter(pts['longitude'], pts['latitude'], c=pts['cluster'], cmap='tab10', s=6, alpha=0.6, edgecolors='none')
plt.colorbar(scatter, ax=ax, label='Cluster ID')
ax.set_title('Global Earthquake Distribution by MLlib K-Means Clusters (k=8)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Longitude (°)', fontsize=12)
ax.set_ylabel('Latitude (°)', fontsize=12)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
plt.tight_layout()
plt.show()


### 3. Top Active Seismic Regions
Bar chart showing the top 12 seismic regions with the highest number of recorded seismic events.


In [ ]:
top_reg = pd.read_csv('../output/top_regions.csv').head(12)
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(top_reg)))
bars = ax.barh(top_reg['region'][::-1], top_reg['quake_count'][::-1], color=colors)
ax.set_title('Top 12 Most Active Seismic Regions by Recorded Event Count', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Recorded Earthquake Count', fontsize=11)
ax.set_ylabel('Region', fontsize=11)
for bar in bars:
    w = bar.get_width()
    ax.annotate(f"{int(w):,}", xy=(w, bar.get_y() + bar.get_height() / 2),
                xytext=(5, 0), textcoords='offset points', ha='left', va='center', fontsize=9)
plt.tight_layout()
plt.show()


### 4. Magnitude Distribution & Temporal Activity Trends


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), dpi=150)

# Magnitude Distribution
mag_dist = pd.read_csv('../output/mag_distribution.csv')
bars = ax1.bar(mag_dist['mag_bucket'], mag_dist['quake_count'], color='#e74c3c', edgecolor='black', alpha=0.85)
ax1.set_yscale('log')
ax1.set_title('Global Magnitude Class Distribution (Log Scale)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Magnitude Bucket', fontsize=10)
ax1.set_ylabel('Event Count (Log Scale)', fontsize=10)
ax1.tick_params(axis='x', rotation=25)

# Yearly Trend
yt = pd.read_csv('../output/yearly_trend.csv')
ax2.plot(yt['year'], yt['quake_count'], marker='o', linewidth=2.5, color='#2980b9')
ax2.fill_between(yt['year'], yt['quake_count'], color='#3498db', alpha=0.2)
ax2.set_title('Yearly Earthquake Frequency Trend', fontsize=12, fontweight='bold')
ax2.set_xlabel('Year', fontsize=10)
ax2.set_ylabel('Recorded Earthquake Count', fontsize=10)
ax2.set_xticks(yt['year'])

plt.tight_layout()
plt.show()


### 5. Interactive Folium Hotspot & Cluster Map
World map featuring seismic density heatmap and cluster centers with detailed popup statistics.


In [ ]:
hg = pd.read_csv('../output/hotspot_grid.csv')
csum = pd.read_csv('../output/cluster_summary.csv')

m = folium.Map(location=[15.0, 0.0], zoom_start=2, tiles='CartoDB Positron')

heat_data = [[row['grid_lat'], row['grid_lon'], min(row['quake_count'], 500)] for _, row in hg.iterrows()]
HeatMap(heat_data, radius=12, blur=15, max_zoom=6, min_opacity=0.35).add_to(m)

colors_list = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'darkgreen']
for _, row in csum.iterrows():
    c_id = int(row['cluster'])
    color = colors_list[c_id % len(colors_list)]
    count_fmt = f"{int(row['quake_count']):,}"
    popup_html = (
        f"<div style='font-family: Arial; font-size: 12px; width: 220px;'>"
        f"<h4 style='margin: 0 0 6px 0; color: #2c3e50;'>Cluster {c_id}</h4>"
        f"<b>Quakes:</b> {count_fmt}<br>"
        f"<b>Avg Depth:</b> {row['avg_depth']} km<br>"
        f"<b>Avg Magnitude:</b> {row['avg_mag']}<br>"
        f"<b>Max Magnitude:</b> {row['max_mag']}<br>"
        f"<b>Top Region:</b> {row['top_region_1']} ({row['top_region_2']})"
        f"</div>"
    )
    folium.Marker(
        location=[row['center_lat'], row['center_lon']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"Cluster {c_id}: {row['top_region_1']} ({count_fmt} quakes)",
        icon=folium.Icon(color=color, icon='info-sign')
    ).add_to(m)

m.save('../output/cluster_map.html')
m
